In [2]:
!pip install mysql-connector-python

In [1]:
pip install google-genai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import pandas as pd
import mysql.connector
from google import genai
from google.genai import types

# Database connection and data extraction
try:
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="INSERT MYSQL PASSWORD", 
        database="marketing_analytics"
    )
    print("Success")
except mysql.connector.Error as err:
    print("Error")

# SQL query to extract aggregated metrics, filtering out 'Unknown' values
query = """
SELECT 
    Customer_Segment, `Group`,
    ROUND(AVG(Sales_Delta), 2) AS avg_revenue_lift,
    ROUND(AVG(Sales_Efficiency) * 100, 2) AS avg_efficiency_pct,
    ROUND(AVG(Satisfaction_Delta), 2) AS avg_satisfaction_lift
FROM marketing_campaign_data
WHERE Customer_Segment != 'Unknown' AND `Group` != 'Unknown'
GROUP BY Customer_Segment, `Group`;
"""

# Load data into a Pandas DataFrame and close the connection
df_metrics = pd.read_sql(query, conn)
conn.close()

# Convert the dataframe to a string format to serve as context for the AI
data_context = df_metrics.to_string(index=False)

# AI Agent configuration
# Initialize the official Google GenAI Client
client = genai.Client(api_key='INSERT API KEY')  # Replace with your Gemini API key

agent_system_prompt = """
You are an expert Agentic AI Data Analyst specializing in Digital Marketing and Growth Strategy.
Your goal is to autonomously review raw A/B testing database results, detect business critical anomalies 
(like satisfaction drops in specific segments), and write a concise, high-impact executive summary for the Chief Marketing Officer (CMO).
Focus heavily on ROI, revenue uplifts, and customer retention risks.
"""

agent_user_prompt = f"""
Analyze the following processed A/B test metrics extracted from our database:

{data_context}

Provide a structured markdown report covering:
1. Executive Summary (Overall performance & ROI validation, referencing the overall 375.6% ROI baseline)
2. Segment Breakdown (Winning segments based on revenue lift)
3. Friction Points (Anomalies or risks regarding customer satisfaction, paying close attention to the Medium Value segment)
4. Strategic Recommendations for the next campaign iteration.
"""

# Execution

print("AI Agent (Gemini) is analyzing the database metrics...")

# Using the gemini-2.5-flash model
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=agent_user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=agent_system_prompt,
        temperature=0.2,
    ),
)

# Extract the text from the Gemini response
report_output = response.text

# Save the report as a Markdown file in your project folder
output_filename = "ai_marketing_report.md"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(report_output)

print(f"Report generated by the AI Agent and saved as '{output_filename}'!")

Success


C:\Users\stefa\AppData\Local\Temp\ipykernel_8996\2796506.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_metrics = pd.read_sql(query, conn)


AI Agent (Gemini) is analyzing the database metrics...
Report generated by the AI Agent and saved as 'ai_marketing_report.md'!
